<a href="https://colab.research.google.com/github/100522401/P1_AA_100522396_100522401/blob/main/notebooks/Cuaderno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practica 1 NIA 100522396 NIA 100522401


# 1. Introducción



El objetivo de la práctica es **construir un modelo** de aprendizaje automático capaz de predecir si un cliente bancario suscribirá un depósito a plazo. Para construir ese modelo nos basaremos en la información demográfica, financiera e interacciones de los clientes.

Como variable objetivo del problema tenemos 'deposit'. Esta variable indicará si el cliente contrató, o no, el producto ofrecido. Por tanto, se trata de un problema de **clasificación binaria**.
<br><br>
En este cuaderno se incluye:

  - Exploratory Data Analysis (EDA).
  - Definir una estrategia de evaluación.
  - Optimización de hiperparámetros (HPO).
  - Selección del mejor modelo y estudio de su rendimiento.
  - Entrenamiento del modelo final.

# 2. Configuración inicial y carga de datos

En esta sección se importan las librerías básicas usadas, se fija la semilla con el nía 1005224**01** y se cargan los datos desde Google Drive

En primer lugar, se importan las librerías básicas y fijamos la semilla aleatoria.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 100522401
np.random.seed(SEED)


Cargamos el dataset de entrenamiento desde Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/P1_AA_datasets"
TRAIN_PATH = f"{BASE_DIR}/bank_01.pkl"
# COMP_PATH  = f"{BASE_DIR}/bank_competition.pkl"

print("TRAIN_PATH:", TRAIN_PATH)
# print("COMP_PATH:", COMP_PATH)

Se comprueba que el dataset se ha cargado correctamente

In [ ]:
#TODO: Se podrían añadir try/excepts al read...

data = pd.read_pickle(TRAIN_PATH)
display(data.head())
print("\nDatos cargados correctamente.")

# 3. EDA

En esta sección se realiza un análisis exploratorio de los datos (EDA) básico.

En este estudio determinaremos:

- Número de instancias.
- Número de variables ( incluyendo la variable objetivo).
- Identificación de variables numéricas, categóricas u ordinales.
- Estudio de cardinalidad.
- Análisis de la variable 'pdays'.
- Estudio de los valores faltantes, constantes o identificadores.
- Estudio del problema y análisis de desbalanceos de la variable objetivo.


## 3.1 Tamaño del dataset y tipo de problema

En este bloque se calcula el tamaño del dataset (número de instancias y variables). También se verfiica que la variable objetivo existe. Se comprueba cuántos valores únicos tienen deposit para confirmar el tipo de problema.

In [ ]:
#Tamaño del dataset y tipo de problema
n_rows, n_cols = data.shape
print(f"Instancias: {n_rows}")
print(f"Variables totales (incluyendo target): {n_cols}")

target = "deposit"
if target not in data.columns:
    raise ValueError(f"No existe la variable objetivo '{target}' en el dataset.")

n_unique_target = data[target].nunique(dropna=False)
print(f"Variable objetivo: {target}")
print(f"Valores únicos en target: {n_unique_target}")

# La variable 'deposit' indica si suscribe o no: problema de clasificación binaria.
if n_unique_target == 2:
    problem_type = "clasificación binaria"
else:
    problem_type = "clasificación (revisar codificación de clases)"

print(f"Tipo de problema: {problem_type}")

En este caso tenemos dos clases, y por lo tanto se concluye que es un problema de **clasificación binaria** (se suscribe o no se suscribe). Es importante tener esto en cuenta ya que condiciona la elección de modelos y métricas en el resto de la práctica.

## 3.2 Tipos de variables

En este bloque se identifican los tipos de variables, separandolas en numéricas (donde se incluyen als booleanas) y categóricas. Esta distinción nos ayudará a definir el preprocesado, ya que las variables numéricas y categóricas requieren transformaciones distintas.

In [ ]:
# 2) Tipos de variables
X = data.drop(columns=[target])

num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Numéricas: {len(num_cols)}")
print(num_cols)
print("\nCategóricas: {0}".format(len(cat_cols)))
print(cat_cols)

Se han identificado 7 variables numéricas y 9 variables catagóricas. Esta separación nos sirve para construir las pipelines de preprocesado, aplicando transformaciones especificas a cada tipo de variable.

## 3.3 Cardinalidad en variables categóricas

Ahora calcularemos la cardinalidad (número de valores únicos) de las variables categóricas para detectar posibles casos de alta cardinalidad. Es importante tenerlo en cuenta porque al aumentar la dimensionalidad del conjunto de datos, aumenta el coste computacional y el rendimiento del modelo.

In [ ]:
# 3) Cardinalidad en variables categóricas
if len(cat_cols) == 0:
    print("No hay variables categóricas.")
else:
    cardinality = X[cat_cols].nunique(dropna=False).sort_values(ascending=False)
    display(cardinality.to_frame("n_unique"))

    high_card_cols = cardinality[cardinality > 10].index.tolist()
    if high_card_cols:
      print("\nCategóricas con alta cardinalidad (>10):", high_card_cols)
    else:
      print("\nCategóricas con alta cardinalidad (>10): ninguna")

En este caso, se observa que hay dos columnas que superan el umbral de alta caridnalidad (> 10 valores únicos).

## 3.4 Valores faltantes

A continuación analizaremos los valores faltantes del dataset, calculando para cada variable cuántos valores nulos aparecen y qué porcentaje representan. Esta información es importante para decidir quÉ estrategia se utilizará en el preprocesado (imputación)

In [ ]:
# 4) Valores faltantes
missing_count = data.isna().sum().sort_values(ascending=False)
missing_ratio = (missing_count / len(data) * 100).round(2)
missing_df = pd.DataFrame({"n_missing": missing_count, "pct_missing": missing_ratio})
missing_df = missing_df[missing_df["n_missing"] > 0]

if missing_df.empty:
    print("No hay valores faltantes.")
else:
    display(missing_df)


Se observa que hay dos varibales con valores faltantes. Estas variables requerirán una estrategia de imputación en el preprocesado, que constará en el pipeline. Esto evitará problemas durante el entrenamiento de los modelos.

## 3.5 Columnas constantes o tipo ID

Se buscan posibles columnas problemáticas para el modelado: columnas constantes y columnas tipo ID (con valores casi únicos o identificables). ESte tipo de variables aporta poca capacidad predicitiva y puede introducir ruido o sobreajusnte, por lo que es mejor detectarlas antes del preprocesado.

In [ ]:
# 5) Columnas constantes y posibles ID
nunique = X.nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()

id_like_by_name = [c for c in X.columns if "id" in c.lower()]
id_like_by_uniqueness = [c for c in X.columns if X[c].nunique(dropna=False) >= 0.98 * len(X)]
id_like_cols = sorted(set(id_like_by_name + id_like_by_uniqueness))

print("Columnas constantes:", constant_cols if constant_cols else "ninguna")
print("Posibles columnas ID:", id_like_cols if id_like_cols else "ninguna")

No se han detectado **columnos constantes** ni **posibles columnas tipo ID** en el dataset. Por tanto, no es necesario eliminar variables por estos motivos en el preprocesado y modelado.

## 3.6 Desbalanceo del target

Ahora analizaremos la distribución de la variable objetivo **deposit**, ya que en problemas de clasificación el desbalanceo relevante relamente es el de las clases que se buscan predecir. Esta información nos será útil para justificar la elección de la métrica y la comparación entre modelos.

En este bloque se analiza la variable **pdays**, la cual representa el númerod e dúas desde el último contacto en la campaña anterior y tomará el valor **-1** cuando el cliente no ha sido contactado previamente. ENtonces, calcularemos la proporcion de valores -1 y se describen los valores válidos (pdays!=-1) para decidir como tratar la variable en el preprocesado.

In [ ]:
# Desbalanceo del target

counts = data[target].value_counts(dropna=False)
perc = data[target].value_counts(normalize=True, dropna=False) * 100

# Conteo y porcentaje por clase
print("Conteo por clase:")
print(counts)

print("\nPorcentajes (%):")
print(perc.round(2))

# Ratio de desbalanceo (mayor/minor)
imbalance_ratio = counts.max() / counts.min()
print("\nImbalance ratio (mayor/minor):", round(imbalance_ratio, 2))

# Gráfico
ax = counts.plot(kind="bar")
plt.title("Distribución del target (deposit)")
plt.xlabel("Clase")
plt.ylabel("Frecuencia")
plt.xticks(rotation=0)
plt.show()

La variable objetivo presenta una distribución de 52,55% para la clase no y 47,45% para la clase yes. Por tanto, el problema puedoe considerarse prácticamente equilibrado, con un desbalanceo poco notorio.

## 3.7 Análisis pdays

In [ ]:
# Analisis de pdays
p = data["pdays"]

n_total = len(p)
n_minus1 = (p == -1).sum()
pct_minus1 = 100 * n_minus1 / n_total

print(f"Total: {n_total}")
print(f"pdays = -1 (sin contacto previo): {n_minus1} ({pct_minus1:.2f}%)")

p_valid = p[p != -1]
print("\nResumen pdays (solo contactos previos reales):")
n_notminus1 = (p != -1).sum()
pct_notminus1 = 100 * n_notminus1 / n_total
print(f"pdays != -1 (solo contactos previos reales): {n_notminus1} ({pct_notminus1:.2f}%)")

display(p_valid.describe())



## TODO, modificar esto, metido en pipeline

Se observa que **pdays == -1** aparece en **8203 de 11000 instancias (74.57%)**, lo que indica que en al mayoría de los casos no existe un contacto previo. Por tanto, el valor -1 no debe interpretarse como un valor numérico, ya que no aporta informaciónr relevante al modelado. Para los casos de **pdays != -1**, el número de días tiene una distribución amplia ( mediana **182**, rango **1 - 854**), lo que aporta información que puede ser útil para el modelado.

En este bloque se dine el preprocesado de **pdays**. Dado que **pdays == -1** no representa un número de días real, sino "sin contacto previo", separemos esta información en una **variable binaria (contacted_before)** y se trasnforma pdays para que contenga únicamente valores numéricos válidos, reemplazando -1 por NaN. De este modo se evita que -1 distorsione el escalado de la variable.

In [ ]:
# # Preprocesado pdays

def transform_pdays(X):
  X = X.copy()
  X["contacted_before"] = (X["pdays"] != -1).astype(int)
  X["pdays"] = X["pdays"].replace(-1, np.nan)
  return X

print("Función transform_pdays definida correctamente.")

## 3.8 Conclusiones EDA TODO

# 4. Estrategia de evaluación

## 4.1 Evaluación outer




En este bloque se define la **evaluación outer** de la práctica. Para ello, se separan las variables predictoras (`X`) y la variable objetivo (`y`). A continuación se realiza una partición **holdout**, reservando **2/3 de los datos para entrenamiento y 1/3 para test**.

El conjunto de test quedará reservado hasta el final de la práctica, lo que nos permitirá obtener un aestimación sobre el rendimiento del modelo sobre datos no vistos.

Además, en este bloque se realiza una **modificación sobre la variable objetivo** (`deposit`) ya que exisdten modelos que así lo requieren. La variable pasa de ser una variabe `yes/no` a ser una **variable binaria** `1/0`. Más concretamente `no -> 0` y ` yes -> 1` . Esta modificación no cambia el significado del problema, sino únicamente la representación de la variable objetivo.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

#Todas las columnas menos la target
X = data.drop(columns='deposit')
#Solo columna target
y = data['deposit']

# Convertir target a variable binaria
y = y.map({"no": 0, "yes": 1})

# Hacemos particiones
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=1/3,
    stratify=y,
    random_state=SEED
)

print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test:", X_test.shape)

print("Distribución de clases en y_train (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nDistribución de clases en y_test (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))

## 4.2 Métrica principal

En esta práctica se utilizará **F1-score** como métrica principal para comparar modelos y seleccionar los hiperparámetros. Se elige esta métrica pese a tener un problema con un desabalanceo leve ya que resume en un único valor el equilibrio entre **precision** y **recall**. Esto permitirá que el modelo **no desperdiciar recursos llamando a cualquiera (Precision) y no perder clientes potenciales (Recall)**

Además del F1-score, se añadiran métricas complementarias:


*   **Accuracy**: al tratarse de un problema relativamente equilibrado, aporta información que puede ser útil sobre el porcentaje de aciertos del modelo.

Esta métrica fue candidata para ser métrica principal. Sin embargo, si frente a datasets desbalanceados se utiliza un modelo ciego que elija la clase mayoritaria, se obtendría como resultado un gran accuracy.

Por lo tanto, de un modelo *Dummy* se obtendrían resultados muy positivos que ignorarían por completo los datos del problema.
*   **ROC AUC**: resulta útil como métrica complemetaria ya que mide la capacidad del modelo para separar las distintas clases.
* **Precision**: para valorar la fiabilidad de las predicciones positivas.
* **Recall**: para medir la capacidad del modelo de detectar casos positivos reales.

De esta manera, el **F1-score** será la referencia principal en la comparación entre alternativas, mientras que el resto de métricas se utilizarán como apoyo a la hora de interpretar resultados.



In [ ]:
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score

# Métrica principal
f1_scorer = make_scorer(f1_score, pos_label=1)

# Métricas complementarias
SCORING = {
    "f1": make_scorer(f1_score, pos_label=1),
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label=1),
    "recall": make_scorer(recall_score, pos_label=1),
    "roc_auc": "roc_auc"
}

## 4.3 Evaluación inner

La evaluación interna **inner** se realizará mediante una **validación cruzada estratificada de 3 particiones** sobre el conjunto de entrenamiento `X__trai` `Ỳ_train`. Se hará uso de esta estrategia para:
* Comparar modelos
* Seleecionar el mejor scaler
* Ajustar hiperparámetros (HPO)

In [ ]:
# Se define la validación cruzada (3-fold)
cv_inner = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)

print('\n')
print(cv_inner)

Se utilizará el F1-score como métrica principal de evaluación, ya que se trata de un problema de clasificación binaria y nos interesa medir de forma equilibrada la capacidad del modelo para identificar correctamente la clase positiva. Aunque el desbalanceo no es muy significativo, la accuracy puede ser menos representativa, mientras que el F1-score combina precisión y recall en una única medida y permite comparar mejor el rendimiento de los distintos modelos.

# 5. Pipeline de preprocesado


## 5.1 Preprocesado

En esta sección se define la pipeline de preprocesado que se aplicará a los datos antes de entrenar cada modelo. La pipeline se encargará de realizar las transformaciones adecuadas para cada tipo de variable garantizando que estas se realicen de manera consistente. Más concretamente, el pipeline se encargará de:
* **Variables numéricas**: Se tratarán los valores faltantes mediante imputación con la mediana. También, en caso de ser necesario, se usará escalado de las variables.

*  **Variables categórcas**: Se tratarán los valores faltantes con la categoría más frecuente o moda. A continuación se codificará mediante **One-Hot Encoding**.

* **Variable** `pdays`: como se ha ha estudiado en el EDA, el valor `-1` no representa el número de días real, sino ausencia de contacto previo. Por tanto se aplicará un tratamiento específico dentro de la pipeline.

## 5.2 Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

def build_preprocessor(num_cols, cat_cols, numeric_scaler=None):
    """Construye el preprocesado para variables numericas y categoricas."""

    # Pipeline para las variables numéricas: imputar NaN con mediana + scaler
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if numeric_scaler is not None:
        numeric_steps.append(("scaler", numeric_scaler))
    numerical_pipeline = Pipeline(numeric_steps)

    # Pipeline para categóricas: imputar NaN con moda + OneHotEncoder
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    # ColumnTransformer: Se aplican las transformaciones según el tipo de variable
    preprocessor = ColumnTransformer([
        ("num", numerical_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)
    ])

    return preprocessor


def create_pipeline(model, num_cols, cat_cols, numeric_scaler=None):
    """Une el preprocesado y el modelo final en una sola pipeline."""

    preprocessor = build_preprocessor(
        num_cols,
        cat_cols,
        numeric_scaler=numeric_scaler
    )

    return Pipeline([
        ("transform_pdays", FunctionTransformer(transform_pdays)),
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])


print("Pipeline base de preprocesado definida correctamente.")


En este bloque se define una función para construir pipelines. La pipeline integra el tratamiento específico de `pdays`, la imputación de valores faltantes,la codificación de variables categóricas y, en caso de ser necesarios, el escalado de variables numéricas.

##5.3 Preparación de columnas

Al definir el preprocesado, las variables numéricas y categóricas se identifican a partir del conjunto de entrenamiento.

Además, como la variable `contacted_before` se genera dentro de la pipeline a partir de `pdays`, es necesario añadirla manualmente a la lista de variables numéricas para poder recibir el tratamiento adecuado en el `ColumnTransformer`.

In [ ]:
num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

if "contacted_before" not in num_cols:
    num_cols = num_cols + ["contacted_before"]

# 6. Métodos básicos: KNN y árboles

## 6.1 Selección del mejor scaler para KNN

En esta sección se comparan los diferentes scalers propuestos para determinar cual esd el más adecuado para **KNN** en este problema. Más concretamente se evaluarán el **StandardScaler**, **MinMaxScaler** y **RobustScaler** usando **KNN con hiperparametros por defecto**.

La comparación entre las distintas alternativas se realiza siguiendo la estrategia de evaluación definida anteriormente, tomando el **F1-score-medio** como criterio principal y utilizando el resto de métrica como apoyo a la hora de interpretar los resultados.


In [ ]:
import time
from sklearn.model_selection import cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Se definen los scalers a comparar
scalers = {
    "standard": StandardScaler(),
    "minmax": MinMaxScaler(),
    "robust": RobustScaler(),
}

# Lista donde se almacenarán los resultados
scaler_results = []

# 3-fold crossvalidation scalers
for scaler_name, scaler in scalers.items():
  # Se crea la pipeline con KNN y scaler actual
    knn_pipe = create_pipeline(
        KNeighborsClassifier(),
        num_cols=num_cols,
        cat_cols=cat_cols,
        numeric_scaler=scaler
    )

    start = time.perf_counter()

    cv_out = cross_validate(
        knn_pipe,
        X_train,
        y_train,
        cv=cv_inner,
        scoring = SCORING,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise"
    )
    elapsed = time.perf_counter() - start

    # Se guardan las medias y desviaciones de las métricas
    scaler_results.append({
        "scaler": scaler_name,
        "f1_mean": cv_out["test_f1"].mean(),
        "f1_std": cv_out["test_f1"].std(),  # Desviación estándar f1-score
        "accuracy_mean": cv_out["test_accuracy"].mean(),
        "precision_mean": cv_out["test_precision"].mean(),
        "recall_mean": cv_out["test_recall"].mean(),
        "roc_auc_mean": cv_out["test_roc_auc"].mean(),
        "fit_time_mean": cv_out["fit_time"].mean(),
        "score_time_mean": cv_out["score_time"].mean(),
        "total_cv_time": elapsed
    })

# Se muestran los resultados
scaler_results_df = pd.DataFrame(scaler_results).sort_values("f1_mean", ascending=False).reset_index(drop=True)
display(scaler_results_df)

best_scaler_name = scaler_results_df.loc[0, "scaler"]
best_scaler = scalers[best_scaler_name]
print(f"\nMejor scaler para KNN según F1 medio: {best_scaler_name}")

De esta comparación se puede concluir que **StandardScaler** es el scaler que ofrece el mejor rendimiento para KNN, ya que obtiene los valores más altos en todas las métricas evaluadas.

Se observa también que **RobustScaler** ocupa una posición intermedia, con un rendimiento inferior al de StandardScaler pero superior al de MinMaxScaler. Por último, **MinMaxScaler** es el scaler que presenta los peores resultados.

Consecuentemente y a partir de este punto, el scaler seleccionado será **StandardScaler**, y por tanto, se empleará cuando el modelo necesite ser escalado.


Para recoger los resultados de los modelos para más tarde compararlos, se crea la siguiente función.

In [ ]:
def add_result(results_list, method, configuration, f1_mean, elapsed):
    results_list.append({
        "Método": method,
        "Configuración": configuration,
        "F1 medio": f1_mean,
        "Tiempo total (s)": elapsed
    })

model_results = []

## 6.2 KNN con hiperparámetros por defecto

En esta sección se evalúa un modelo **KNN** con sus hiperparámetros por defecto, utilizando el método de escalado ya seleccionado(**StandardScaler**). EL objetivo es obtener una referencia inicial del rendimiento antes de realizar la HPO.

Además se medirá el tiempo de entrenamiento, ya que el coste computacional tendrá también peso en la comparación.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
import time

# Inner crossvalidation
cv_inner = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)

# Pipeline
pipeline_knn_default = create_pipeline(
    model=KNeighborsClassifier(),
    num_cols=num_cols,
    cat_cols=cat_cols,
    numeric_scaler=StandardScaler()
)

# Tiempo
start = time.perf_counter()

scores_knn_default = cross_validate(
    pipeline_knn_default,
    X_train,
    y_train,
    cv=cv_inner,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise"
  )

elapsed = time.perf_counter() - start

print("KNN con hiperparámetros por defecto:\n")
print(f"F1 medio: {scores_knn_default['test_f1'].mean():.4f} ± {scores_knn_default['test_f1'].std():.4f}")
print(f"\nAccuracy medio: {scores_knn_default['test_accuracy'].mean():.4f}")
print(f"Precision media: {scores_knn_default['test_precision'].mean():.4f}")
print(f"Recall medio: {scores_knn_default['test_recall'].mean():.4f}")
print(f"ROC-AUC medio: {scores_knn_default['test_roc_auc'].mean():.4f}")
print(f"Tiempo medio de fit: {scores_knn_default['fit_time'].mean():.4f} s")
print(f"Tiempo total CV: {elapsed:.4f} s")

add_result(model_results, "KNN", "por defecto", scores_knn_default["test_f1"].mean(), elapsed)

##6.3 HPO de KNN
Una vez ya se ha evaluado el KNN con sus hiperparámetros por defecto, en esta subsección se ajustan sus hiperparámetros mediante un `Grid Search` o búsqueda en rejilla y validación cruzada de 3 particiones. El objetivo es comprobar si es posible mejorar el rendimiento del modelo y analizar si la mejora compensa el posible aumento del coste computacional.


In [ ]:
from sklearn.model_selection import GridSearchCV

# Pipeline KNN
pipeline_knn = create_pipeline(
    model=KNeighborsClassifier(),
    num_cols=num_cols,
    cat_cols=cat_cols,
    numeric_scaler=StandardScaler()
)

# Grid
param_grid_knn = {
    "classifier__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
    "classifier__weights": ["uniform", "distance"],
    "classifier__p": [1, 2]
}

grid_knn = GridSearchCV(
    estimator=pipeline_knn,
    param_grid=param_grid_knn,
    scoring=f1_scorer,
    cv=cv_inner,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise"
)

#Tiempo
start = time.perf_counter()

grid_knn.fit(X_train, y_train)

elapsed = time.perf_counter() - start

#Resultados
print("KNN con HPO")
print("Mejores hiperparámetros para KNN:", grid_knn.best_params_)
print(f"Mejor F1 medio: {grid_knn.best_score_:.4f}")
print(f"Tiempo total HPO: {elapsed:.4f} segundos")

add_result(model_results, "KNN", "HPO", grid_knn.best_score_, elapsed)

Tras la optimización de hiperparámetros, el mejor modelo KNN se obtiene con **11 neighbors**, distancia de **Minkowski** con `p = 2`, que es equivalente a la **distancia euclídea** y la ponderación por distancia (`weights = "distance"`). Por tanto, con esta configuración se obtiene que el modelo alcanza un **F1-score medio de 0.7844**.

Conparado con el **KNN por defecto**, podemos observar una **ligera mejora** en el rendimiento, ya que el F1-score pasa de 0.775 a 0.784 aproximadamente. Esto indica que el ajuste de hiperparámetros sí aporta una ganancia, pero esta no es muy grande. Sin embargo, esta mejora implica un **coste computacional considerablemente mayor**, ya que el proceso de HPO ha requerido **45.33 segundos**.

En cuanto a las conclusiones de la mejor configuración, el hecho de que esta use 11 neighbours sugiere que, en este caso, KNN funciona mejor con un número de vecinos mediano. Además, es uso de distancias indica que resulta beneficioso dar más importancia a los vecinos más cercanos.

## 6.4 Análisis del efecto de los hiperparámetros en KNN

En esta sección se analiza cómo afectan los principales hiperparámetros de KNN al rendimiento del modelo. Concretamente se estudia el efecto del número de vecinos (`n_neighbors`), la ponderación (`weights`) y el parámetro `p`de la distancia de Minkowski.

Dadp que la mejor configuración obtenida utiliza `weights = "distance"`, `p=2`y `n_neighbors = 11`, en este apartado se irán fijando los parametros de dicha configuración para analizar de forma aislada cómo varía el rendimiento del modelo.

En primer lugar se preparan los resultados obtenidos tras realizar el `GridSearch`.

In [ ]:
results_knn = pd.DataFrame(grid_knn.cv_results_)

results_knn_plot = results_knn[[
    "param_classifier__n_neighbors",
    "param_classifier__weights",
    "param_classifier__p",
    "mean_test_score",
    "std_test_score",
    "mean_fit_time",
    "rank_test_score"
]].copy()

results_knn_plot = results_knn_plot.sort_values("mean_test_score", ascending=False)
display(results_knn_plot.head(10))

A continuación se muestra el plot que compara `uniform`y `distance`fijando el parámetro `p=2`.

In [ ]:
plt.figure(figsize=(8,5))

for weight_type in results_knn["param_classifier__weights"].unique():
    subset = results_knn[
        (results_knn["param_classifier__weights"] == weight_type) &
        (results_knn["param_classifier__p"] == 2)
    ].sort_values("param_classifier__n_neighbors")

    plt.plot(
        subset["param_classifier__n_neighbors"],
        subset["mean_test_score"],
        marker="o",
        label=f"weights = {weight_type}"
    )

plt.title("Efecto de n_neighbors en KNN (p = 2)")
plt.xlabel("Número de vecinos")
plt.ylabel("F1-score medio")
plt.legend()
plt.grid(True)
plt.show()

Se observa que ambas gráficas tienen una forma similar, pero que la gráfica referente `weights = "distance"` por lo general obtiene un **mayor F1-score medio**.

Ahora se compararán los resultados obtenidos con `p=1`y con `p=2`fijando esta vez `weights = "distance"`.

In [ ]:
plt.figure(figsize=(8,5))

for p_value in results_knn["param_classifier__p"].unique():
    subset = results_knn[
        (results_knn["param_classifier__p"] == p_value) &
        (results_knn["param_classifier__weights"] == "distance")
    ].sort_values("param_classifier__n_neighbors")

    plt.plot(
        subset["param_classifier__n_neighbors"],
        subset["mean_test_score"],
        marker="o",
        label=f"p = {p_value}"
    )

plt.title("Efecto de p en KNN (weights = distance)")
plt.xlabel("Número de vecinos")
plt.ylabel("F1-score medio")
plt.legend()
plt.grid(True)
plt.show()

En este caso se observa un claro mejor rendimiento con `p=2`.

Para concluir, se observa cómo el rendimiento del modelo varía con el número de vecinos. EN general, valores muy bajos de `n_neighbors` hacen al modelo más sensible al ruido (favorece el **Overfitting**), mientras que valores más altos hacen que las fronteras de decisión sean muy suaves (puede provocar **Underfitting**). En este problema, el mejor comportamiento se alcanza con un número de vecinos intermedio.

##DummyClassifier
// este apartado luego lo movemos al final para las conclusiones

En esta sección se evalúan dos clasificadores **Dummy** para medir el rendimiento de clasificadores **triviales**. Un **DummyClassifier** no aprende patrones reales de los atributos, simplemente predice según una regla muy básica, por ejemplo escoger siempre la clase mayoritaria o generar predicciones aleatorias respetando la distribución de clases.

Esto sirve como base. Si **KNN** o los **árboles** no mejoran claramente a un **dummy**, entonces el modelo no estaría aprendiendo información útil del problema.

In [ ]:
from sklearn.dummy import DummyClassifier

# Se definen los clasificadores dummy a comparar
dummy_models = {
    "most_frequent": DummyClassifier(strategy="most_frequent"),
    "stratified": DummyClassifier(strategy="stratified", random_state=SEED),
}

# Lista donde se guardan los resultados
dummy_results = []

# Evaluación de cada dummy con validación cruzada
for dummy_name, dummy_model in dummy_models.items():
    start = time.perf_counter()

    cv_out = cross_validate(
        dummy_model,
        X_train,
        y_train,
        cv=cv_inner,
        scoring=SCORING,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise"
    )

    elapsed = time.perf_counter() - start

    # Se guardan las medias de las métricas y los tiempos
    dummy_results.append({
        "modelo": dummy_name,
        "f1_mean": cv_out["test_f1"].mean(),
        "f1_std": cv_out["test_f1"].std(),
        "accuracy_mean": cv_out["test_accuracy"].mean(),
        "precision_mean": cv_out["test_precision"].mean(),
        "recall_mean": cv_out["test_recall"].mean(),
        "roc_auc_mean": cv_out["test_roc_auc"].mean(),
        "fit_time_mean": cv_out["fit_time"].mean(),
        "score_time_mean": cv_out["score_time"].mean(),
        "total_cv_time": elapsed
    })



# Se muestran los resultados ordenados por F1 medio
dummy_results_df = pd.DataFrame(dummy_results).sort_values("f1_mean", ascending=False).reset_index(drop=True)
display(dummy_results_df)

best_dummy_row = dummy_results_df.iloc[0]

add_result(model_results, "Dummy",
    best_dummy_row["modelo"],
    best_dummy_row["f1_mean"],
    best_dummy_row["total_cv_time"]
)

##6.5 Árboles con hiperparámetros por defecto

A continuación se evalúa un **DecisionTreeClassifier** con sus hiperparámetros por defecto. En este caso, a diferencia de KNN, no es necesario aplicar escalado a las variables numéricas, ya que los árboles no dependen de distancias entre observaciones.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Pipeline
pipeline_tree_default = create_pipeline(
    model=DecisionTreeClassifier(random_state=SEED),
    num_cols=num_cols,
    cat_cols=cat_cols,
    numeric_scaler=None
)

# Tiempo
start = time.perf_counter()

scores_tree_default = cross_validate(
    pipeline_tree_default,
    X_train,
    y_train,
    cv=cv_inner,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise"
)

elapsed = time.perf_counter() - start

print("Árbol con hiperparámetros por defecto:\n")
print(f"F1 medio: {scores_tree_default['test_f1'].mean():.4f} ± {scores_tree_default['test_f1'].std():.4f}")
print(f"\nAccuracy medio: {scores_tree_default['test_accuracy'].mean():.4f}")
print(f"Precision media: {scores_tree_default['test_precision'].mean():.4f}")
print(f"Recall medio: {scores_tree_default['test_recall'].mean():.4f}")
print(f"ROC-AUC medio: {scores_tree_default['test_roc_auc'].mean():.4f}")
print(f"Tiempo medio de fit: {scores_tree_default['fit_time'].mean():.4f} s")
print(f"Tiempo total CV: {elapsed:.4f} s")

add_result(model_results, "Árbol", "por defecto", scores_tree_default["test_f1"].mean(), elapsed)

##6.6 Árbol poco profundo

Además del árbol con hiperparámetros por defecto, utilizamos un árbol **poco profundo** para obtener una interpretación de cómo podrían tomarse decisiones en este problema. Para ello se entrenará un árbol con profundidad reducida, lo que facilita visualizar las reglas principales de decisión.

In [ ]:
from sklearn.tree import plot_tree

# Pipeline del árbol poco profundo
pipeline_tree_shallow = create_pipeline(
    model=DecisionTreeClassifier(max_depth=4, random_state=SEED),
    num_cols=num_cols,
    cat_cols=cat_cols,
    numeric_scaler=None
)

# Se ajusta el modelo para poder visualizarlo
pipeline_tree_shallow.fit(X_train, y_train)

# Se recuperan los nombres de las variables tras el preprocesado
preprocessor_fitted = pipeline_tree_shallow.named_steps["preprocessor"]
cat_feature_names = preprocessor_fitted.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(cat_cols)
feature_names = num_cols + list(cat_feature_names)

cv_out_tree_shallow = cross_validate(
    pipeline_tree_shallow,
    X_train,
    y_train,
    cv=cv_inner,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise"
)


# Representación gráfica del árbol
plt.figure(figsize=(24, 10))
plot_tree(
    pipeline_tree_shallow.named_steps["classifier"],
    feature_names=feature_names,
    class_names=["no", "yes"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Árbol de decisión poco profundo (max_depth=4)")
plt.show()

print(f"F1 medio: {cv_out_tree_shallow['test_f1'].mean():.4f}")
print(f"Accuracy media: {cv_out_tree_shallow['test_accuracy'].mean():.4f}")


## 6.7 HPO de Árboles
Una vez ya se ha evaluado el árbol con sus hiperparámetros por defecto, en esta subsección se ajustan sus hiperparámetros mediante un `Grid Search` y validación cruzada de 3 particiones. El objetivo es comprobar si es posible mejorar el rendimiento del modelo y analizar si la mejora compensa el posible aumento del coste computacional.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Pipeline del árbol
pipeline_tree = create_pipeline(
    model=DecisionTreeClassifier(random_state=SEED),
    num_cols=num_cols,
    cat_cols=cat_cols,
    numeric_scaler=None
)

# Grid
param_grid_tree = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [3, 5, 7, 10, None],
    "classifier__min_samples_split": [2, 10, 20, 50],
    "classifier__min_samples_leaf": [1, 5, 10, 20]
}

grid_tree = GridSearchCV(
    estimator=pipeline_tree,
    param_grid=param_grid_tree,
    scoring=f1_scorer,
    cv=cv_inner,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise"
)

# Tiempo
start = time.perf_counter()

grid_tree.fit(X_train, y_train)

elapsed = time.perf_counter() - start

# Resultados
print("Trees con HPO")
print("Mejores hiperparámetros para Trees:", grid_tree.best_params_)
print(f"Mejor F1 medio: {grid_tree.best_score_:.4f}")
print(f"Tiempo total HPO: {elapsed:.4f} segundos")

add_result(model_results, "Árbol", "HPO", grid_tree.best_score_, elapsed)

 Tras la optimización de hiperparámetros, el mejor árbol se obtiene con criterion = "gini", max_depth = 10, min_samples_leaf = 5 y min_samples_split = 50, alcanzando un **F1-score medio de 0.8177**. En comparación con el árbol por defecto, esto supone una mejora notable, ya que el F1-score pasa de **0.7734 a *0.8177**. Por tanto, en este caso el ajuste de hiperparámetros sí aporta una ganancia clara de rendimiento. De
  hecho, el árbol ajustado supera también al *KNN optimizado*, por lo que se convierte en la mejor alternativa dentro de los métodos probados hasta el momento.


## 6.8 Análisis del efecto de los hiperparámetros en Trees

En esta sección se analiza cómo afectan los principales hiperparámetros de los árboles al rendimiento del modelo. Concretamente se estudiará el efecto de `max_depth` y `min_samples_leaf`, fijando el resto de parámetros en la mejor configuración obtenida tras el `Grid Search`.

En primer lugar se preparan los resultados obtenidos tras realizar el `GridSearch`.

In [ ]:
results_tree = pd.DataFrame(grid_tree.cv_results_)

results_tree_plot = results_tree[[
    "param_classifier__criterion",
    "param_classifier__max_depth",
    "param_classifier__min_samples_split",
    "param_classifier__min_samples_leaf",
    "mean_test_score",
    "std_test_score",
    "mean_fit_time",
    "rank_test_score"
]].copy()

results_tree_plot = results_tree_plot.sort_values("mean_test_score", ascending=False)
display(results_tree_plot.head(10))


A continuación se muestra el plot que analiza el efecto de `max_depth` fijando el resto de hiperparámetros en la mejor configuración encontrada.

In [ ]:
best_criterion = grid_tree.best_params_["classifier__criterion"]
best_min_samples_split = grid_tree.best_params_["classifier__min_samples_split"]
best_min_samples_leaf = grid_tree.best_params_["classifier__min_samples_leaf"]

subset_depth = results_tree[
    (results_tree["param_classifier__criterion"] == best_criterion) &
    (results_tree["param_classifier__min_samples_split"] == best_min_samples_split) &
    (results_tree["param_classifier__min_samples_leaf"] == best_min_samples_leaf)
].copy()

subset_depth = subset_depth.sort_values(
    by="param_classifier__max_depth",
    key=lambda s: s.apply(lambda x: float("inf") if x is None else x)
)
subset_depth["max_depth_label"] = subset_depth["param_classifier__max_depth"].apply(
    lambda x: "None" if x is None else str(int(x))
)

plt.figure(figsize=(8,5))
plt.plot(
    subset_depth["max_depth_label"],
    subset_depth["mean_test_score"],
    marker="o"
)
plt.title(
    f"Efecto de max_depth en Trees ({best_criterion}, min_split = {best_min_samples_split}, min_leaf = {best_min_samples_leaf})"
)
plt.xlabel("max_depth")
plt.ylabel("F1-score medio")
plt.grid(True)
plt.show()


  En este gráfico se observa que el rendimiento del árbol mejora de forma progresiva a medida que aumenta max_depth, pasando de peores resultados con profundidades bajas a su mejor resultado en **max_depth = 10**. Sin embargo, cuando se elimina la restricción de profundidad (None), el rendimiento desciende ligeramente. Esto nos indica que el árbol necesita cierta complejidad para capturar bien la estructura del problema, pero que dejarlo crecer sin límite introduce sobreajuste. Por tanto, una profundidad intermedia-alta parece ofrecer el mejor equilibrio entre capacidad predictiva y generalización.

In [ ]:
best_max_depth = grid_tree.best_params_["classifier__max_depth"]

subset_leaf = results_tree[
    (results_tree["param_classifier__criterion"] == best_criterion) &
    (results_tree["param_classifier__max_depth"] == best_max_depth) &
    (results_tree["param_classifier__min_samples_split"] == best_min_samples_split)
].sort_values("param_classifier__min_samples_leaf")

depth_label = "None" if best_max_depth is None else str(int(best_max_depth))

plt.figure(figsize=(8,5))
plt.plot(
    subset_leaf["param_classifier__min_samples_leaf"].astype(int),
    subset_leaf["mean_test_score"],
    marker="o"
)
plt.title(
    f"Efecto de min_samples_leaf en Trees ({best_criterion}, max_depth = {depth_label}, min_split = {best_min_samples_split})"
)
plt.xlabel("min_samples_leaf")
plt.ylabel("F1-score medio")
plt.grid(True)
plt.show()


 El análisis de min_samples_leaf muestra que el mejor rendimiento se obtiene con
  *min_samples_leaf = 5*. Con hojas demasiado pequeñas, el resultado es algo peor, lo que sugiere un mayor riesgo de sobreajuste. Por otro lado, al aumentar demasiado este parámetro, el rendimiento vuelve a descender, indicando que el modelo se vuelve demasiado restrictivo. Por tanto, los resultados apuntan a que una regularización moderada
  mediante hojas pequeñas mejora la capacidad de generalización del árbol.

##6.9 Conclusiones


Una vez ya se han entrenado y evaluado los **modelos básicos**, se han resumido los resultados obtenidos. En la siguiente tabla se comparan los valores obtenidos en **F1-score** como el **tiempo total** del proceso asociado. Por lo tanto, en esta tabla se podrá visualizar de forma directa las diferencias de rendimiento y coste computacionar entre los modelos **KNN** y **árboles de decisión**, tanto en sus versiones por defecto como en sus mejores configuraciones (halladas tras hacer el HPO). También se compararán frente a un clasificador **Dummy** (`DummyClassifier`).

In [ ]:
summary_models = pd.DataFrame(model_results)

summary_models["F1 medio"] = summary_models["F1 medio"].round(4)
summary_models["Tiempo total (s)"] = summary_models["Tiempo total (s)"].round(4)

summary_models = summary_models.sort_values("F1 medio", ascending=False).reset_index(drop=True)

display(summary_models)

A partir de la tabla, se observa que la mejor alternativa dentro de este estudio es el **árbol de decisión con HPO**, que alcanza rl mayor **F1-score medio**. En segundo lugar esta el **KNN con HPO**. Seguido quedan las configuraciones por defecto de ambos métodos, que presentan unos resultados muy similares.

En cuanto al HPO, tanto en **KNN** como en **árboles** se aprecia una mejora respecto a la configuración por defecto. No obstante, esta mejora es más notoria en el caso de los **árboles**.

También conviene señalar que, en las configuraciones con **HPO**, el tiempo mostrado corresponde al **proceso completo de HPO** mediante `GridSearchCV`. Es por esto que es considferablemente mayor que el tiempo necesario para ajustar posteriormente un modelo final.

Para concluir, todos los modelos evaluados superan al **DummyClassifier**, lo que confirma que los métodos estan realmente reteniendo información de los datos. De esta manera, dentor de este bloque, la alternativa más competitiva es el **arbol de decisión con HPO**, ya que combina el mejor rendimiento con un coste computacional razonable.

# 7. Modelos avanzados: lineales y SVM

En esta sección se estudian dos tipos de métodos avanzados de clasificación: los **modelos lineales** y las **máquinas de vectores de soporte (SVM)**. Ambos métodos se evaluarán en primer lugar con sus hiperparámetros por dfefecto y, luego, se realizará su HPO.

El objetivo será comprobar si estos métodos son capaces de mejorar los resultados obtenidos en el bloque anterior con **KNN** y **árboles**, asi como analizar su coste computacional y capacidades.

## 7.1 Regresión logística con hipérparámetros por defecto

A continuación se evaluará un **regresión logísita** con sus hiperparámetros por defecto. Este es un modelo lineal para clasificación binaria cuya salida se transforma mediante una función sigmoida, loq ue permite interpretar la predicción como la probabilidad de que pertencezca a al clase positiva.

El objetivo de esta sección es obtener una referencia inicial del rendimiento del modle oantes de realizar la HPO. Tambíen se medirá el tiempo total del proceso para poder comparar posteriormente su coste computacional.

In [ ]:
# Se inicializa nueva tabla comparativa
advanced_model_results = []

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
import time

# Inner crossvalidation
cv_inner = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)

# Pipeline
pipeline_logreg_default = create_pipeline(
    model=LogisticRegression(),
    num_cols=num_cols,
    cat_cols=cat_cols,
    numeric_scaler=StandardScaler()
)

# Tiempo
start = time.perf_counter()

scores_logreg_default = cross_validate(
    pipeline_logreg_default,
    X_train,
    y_train,
    cv=cv_inner,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise"
  )

elapsed = time.perf_counter() - start

print("Modelo lineal (Regresión logística) default:\n")
print(f"F1 medio: {scores_logreg_default['test_f1'].mean():.4f} ± {scores_logreg_default['test_f1'].std():.4f}")
print(f"\nAccuracy medio: {scores_logreg_default['test_accuracy'].mean():.4f}")
print(f"Precision media: {scores_logreg_default['test_precision'].mean():.4f}")
print(f"Recall medio: {scores_logreg_default['test_recall'].mean():.4f}")
print(f"ROC-AUC medio: {scores_logreg_default['test_roc_auc'].mean():.4f}")
print(f"Tiempo medio de fit: {scores_logreg_default['fit_time'].mean():.4f} s")
print(f"Tiempo total CV: {elapsed:.4f} s")

add_result(advanced_model_results, "Regresión logística", "por defecto", scores_logreg_default["test_f1"].mean(), elapsed)

La regresión logística por defecto ofrece una primera referencia ya muy competitiva respecto a los modelos básicos.

También, al tratarse de un método relativamente simple y con bajo coste computacional, puede llegar a ser una alternativa frente a modelo más complejos.

## 7.2 Modelo lineal con HPO

Tras evaluar el model lineal con sus hiperparámetros por defecto, se pocede a realizar su **HPO** con `GridSearchCV`. El objetivo es comprobar si el rendimiento del modelo mejora y en ese caso comprobar si la mejora compensa el coste computacional adicional.

En caso de la regresión logísitca, los hiperparámetros mas relevantes son:
* `C`: Controla la intensidad de la regularización.
* `penalty`: determina el tipo de regularización utilizada. Puede variar entre **L1**, **L2**, ambas (**ElasticNet**) o`ninguna.
* `solver`: Algoritmo empleado para ajustlar los parámetros del modelo.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Pipeline KNN
pipeline_logreg_hpo = create_pipeline(
    model=LogisticRegression(),
    num_cols=num_cols,
    cat_cols=cat_cols,
    numeric_scaler=StandardScaler()
)

# Grid
param_grid_logreg = [
    {
        "classifier__penalty": ["l2"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__solver": ["lbfgs"]
    },
    {
        "classifier__penalty": ["l1"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__solver": ["saga"]
    },
    {
        "classifier__penalty": ["elasticnet"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__solver": ["saga"],
        "classifier__l1_ratio": [0.2, 0.5, 0.8]
    }
]

grid_logreg = GridSearchCV(
    estimator=pipeline_logreg_hpo,
    param_grid=param_grid_logreg,
    scoring=f1_scorer,
    cv=cv_inner,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise"
)

#Tiempo
start = time.perf_counter()

grid_logreg.fit(X_train, y_train)

elapsed = time.perf_counter() - start

# Resultados
print("Regresión logística con HPO")
print("Mejores hiperparámetros:", grid_logreg.best_params_)
print(f"Mejor F1 medio: {grid_logreg.best_score_:.4f}")
print(f"Tiempo total HPO: {elapsed:.4f} segundos")

add_result(advanced_model_results, "Regresión logística", "HPO", grid_logreg.best_score_, elapsed)

Antes de sacar conclusiones, se estudiarán los casos de prueba. En primer lugar, para los valores de `C` se ha escogido una escala logarítmica para poder probar desde una regularización fuerte hasta una débil.

En cuanto a los solvers, se han utilizado únicamente combinaciones compatibles.
* `lbfgs` para `l2`.
* `saga` para `l1` y `elasticnet` ya que permite trabajar con esas regularizaciones y, es necesario para `elasticnet`.

Por último, el parámetro `l1_ratio`  se ha incluido únicamente en `elasticnet` aya que controla el equilibrio entre las penalizaciones L1 y L2.

En cuanto a las conclusiones, se ha obtenido que la configuración óptima es **L2**, con `C = 10 ` y `solver = "lbfgs"`. De este resultado se puede decir que, para este problema una regularización estándar y medianamente suave ses suficiente para obtener el mejor rendimiento dentro de los casos estudiados.

En comparación con la versión por defecto, el HPO produce una **Mejora ligera** del rendimiento. Sin embargo, aunque con el HPO se afine el modelo, la ganancia en cuanto a rendimiento es leve respecto al coste computacional que requiere.

## 7.3 Análisis modelo lineal

A continuación se analizará mediante la regresión logística de que manera influyen las variables en la predicción y en que medida.

Para ello se hará uso de la **mejor versión de la regresión logística** , es decir, la regresión logística tras HPO. A partir de sus coeficientes se podrá identificar que variables favorecen a la clase positiva (`yes`) y cuales favorecen a la clase negativa (`no`).

En primer lugar, se contruirá una tabla con las variables que tienen los coeficientes absolutos más grandes. En esta tabla se identificará que atributos son los más relevantes para el modelo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Mejor pipeline de regresión logística tras HPO
best_logreg = grid_logreg.best_estimator_

# Recuperar nombres de variables tras el preprocesado
preprocessor_fitted = best_logreg.named_steps["preprocessor"]
cat_feature_names = preprocessor_fitted.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(cat_cols)
feature_names = num_cols + list(cat_feature_names)

# Coeficientes del clasificador
coefs = best_logreg.named_steps["classifier"].coef_[0]

coef_df = pd.DataFrame({
    "variable": feature_names,
    "coeficiente": coefs,
    "abs_coef": np.abs(coefs)
}).sort_values("abs_coef", ascending=False)

display(coef_df.head(20))

Para interpretar correctamente esta tabla es importante tener en cuenta que un coeficiente **positivo** favorece al clase `yes`, mientras que un coeficiente **negativo** favorece la clase `no`. Además, cuanto mayor es el valor absoluto del coeficiente, mayor es el peso de esa variable a la hora de influir en las predicciones del modelo.

También, para una correcta interpretación se debe tener en cuenta que las variables segguidas de un `_atributo`, como por ejemplo, `month_oct` son variables categóricas que han sido transformadas mediante **One-Hot Encoding**.

A partir de los resultados, se observa que variables como `education_None`, `job_None`, `month_mar`, `duration`y `poutcome_success`son importantes para la predicción ya que empujan el resultado hacia la clase positiva. Por otro lado variables como `education_primary`, `month_jan`, `education_secondary` o `job_unknown` tienen a decidir el resultado hacia la clase negativa.

Ahora se mostrará un gráfico para visualizar las conclusiones anteriores.

In [ ]:
top_n = 20
top_features = coef_df.head(top_n).sort_values("coeficiente")

plt.figure(figsize=(10, 6))
plt.barh(top_features["variable"], top_features["coeficiente"])
plt.title("Variables más influyentes en la regresión logística")
plt.xlabel("Coeficiente")
plt.ylabel("Variable")
plt.show()

## SVM con hiperparámetros por defecto


En esta sección se evaluará una **máquina de vectores de soporte (SVM)** con sus hiperparámetros por defecto. Este método se resume en buscar una frontera de decisión que separe de la mejor forma (con el mayor margen posible) las clases.

El objetivo es obterner una referencia incial del comportamiento de este modelo antes de proceder con el HPO.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, StratifiedKFold
import time

# Inner crossvalidation
cv_inner = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=SEED
)

# Pipeline
pipeline_svm_default = create_pipeline(
    model=SVC(),
    num_cols=num_cols,
    cat_cols=cat_cols,
    numeric_scaler=StandardScaler()
)

# Tiempo
start = time.perf_counter()

scores_svm_default = cross_validate(
    pipeline_svm_default,
    X_train,
    y_train,
    cv=cv_inner,
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise"
  )

elapsed = time.perf_counter() - start

print("SVM con hiperparámetros por defecto:\n")
print(f"F1 medio: {scores_svm_default['test_f1'].mean():.4f} ± {scores_svm_default['test_f1'].std():.4f}")
print(f"\nAccuracy medio: {scores_svm_default['test_accuracy'].mean():.4f}")
print(f"Precision media: {scores_svm_default['test_precision'].mean():.4f}")
print(f"Recall medio: {scores_svm_default['test_recall'].mean():.4f}")
print(f"ROC-AUC medio: {scores_svm_default['test_roc_auc'].mean():.4f}")
print(f"Tiempo medio de fit: {scores_svm_default['fit_time'].mean():.4f} s")
print(f"Tiempo total CV: {elapsed:.4f} s")

add_result(advanced_model_results, "SVM", "por defecto", scores_svm_default["test_f1"].mean(), elapsed)

Este modelo ya con hiperparámetros por defecto ofrece un rendimiento muy competitivo dentro de los modelos avanzados.

Además, aunque tiene un coste computacional mayor la regresión logística, sigue siendo un coste asumible respecto a su rendimiento.

## 7.5 HPO de SVM

Una vez evaluada la SVM con sus hiperparámetros por defecto, se procederá a ajustar sus principales hiperparámetros mediante un `GridSearchCV`. El objetivo será comprobar si el renimiento del modelo puede mejorar y analizar si esa mejora compensa el aumento del coste computacional.

Hiperparámetros a optimizar:
* `C`: Controla el equilibrio entre maximizar el margen y la penalización de errores.
* `kernel`: es la función que indica la "similitud" entre instancias. Es decir, determina que forma tendrá la frontera de decisión.

* `gamma`: En los kernels no lineales, controla la influencia de cada instancia.

In [ ]:
from sklearn.svm import SVC

# Pipeline KNN
pipeline_svc = create_pipeline(
    model=SVC(),
    num_cols=num_cols,
    cat_cols=cat_cols,
    numeric_scaler=StandardScaler()
)

# Grid
param_grid_svc = [
    {
        "classifier__kernel": ["linear"],
        "classifier__C": [0.01, 0.1, 1, 10, 100]
    },
    {
        "classifier__kernel": ["rbf"],
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__gamma": ["scale", "auto", 0.01, 0.1, 1]
    }
]

grid_svc = GridSearchCV(
    estimator=pipeline_svc,
    param_grid=param_grid_svc,
    scoring=f1_scorer,
    cv=cv_inner,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise"
)

#Tiempo
start = time.perf_counter()

grid_svc.fit(X_train, y_train)

elapsed = time.perf_counter() - start

# Resultados
print("SVM con HPO")
print("Mejores hiperparámetros para SVM:", grid_svc.best_params_)
print(f"Mejor F1 medio: {grid_svc.best_score_:.4f}")
print(f"Tiempo total HPO: {elapsed:.4f} segundos")

add_result(advanced_model_results, "SVM", "HPO", grid_svc.best_score_, elapsed)

En concreto se han comparado dos tipos de **kernel**:
* `lineal`, para  comprobar si una frontera de decisión lineal es suficiente.
* **RBF** o `gaussiano`, para estudiar fronteras no lineales más flexibles.

En ambos casos se ajusta `C`. Sin embargo, en el caso del kernel **gaussiano**, también se ajusta `gamma`.

Tras el HPO, la mejor combinación utiliza un **kernel gaussiano** con `C=100` y `gamma = 0.01`. Esto indica que, para este problema, una frontera de decisión **no lineal** es más adecuada.

En comparación con la versión por defecto, el HPO produce una **mejora muy leve** del rendimiento. Por tanto, aunque el HPO permite afinar el modelo, la ganancia obtenida es muy leve respecto al incremento del aumento del coste computacional tras realizar el proceso completo de HPO.